In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split

from torchvision import datasets, transforms
from PIL import Image

import cv2
import time
import random
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

In [3]:
#config
DATA_DIR      = "./processed_data"
IMG_SIZE      = 96
BATCH_SIZE    = 64
EPOCHS        = 70
LEARNING_RATE = 0.0005
VAL_SPLIT     = 0.2
MODEL_PATH    = "./model.pth"
PATIENCE      = 10          

In [4]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

In [5]:
assert torch.cuda.is_available(), "nema"
device = torch.device("cuda")
print(device)

cuda


In [ ]:
class cnn(nn.module):
    def __init__(self, num_classes):
        super().__init__()
        #ekstrakcija
        self.features = nn.sequential(
            #48x48
            nn.conv2d(3, 32, 3, padding=1),
            nn.batchnorm2d(32), nn.relu(), nn.maxpool2d(2),

            #24x24
            nn.conv2d(32, 64, 3, padding=1),
            nn.batchnorm2d(64), nn.relu(), nn.maxpool2d(2),

            # 12x12
            nn.conv2d(64, 128, 3, padding=1),
            nn.batchnorm2d(128), nn.relu(), nn.maxpool2d(2),

            #6x6
            nn.conv2d(128, 256, 3, padding=1),
            nn.batchnorm2d(256), nn.relu(), nn.maxpool2d(2),

            #3x3
            nn.conv2d(256, 512, 3, padding=1),
            nn.batchnorm2d(512), nn.relu(), nn.maxpool2d(2),
        )
        # klasifikacija
        self.classifier = nn.sequential(
            nn.adaptiveavgpool2d(1),   #3x3 u 1x1
            nn.flatten(),              #vektor od 512
            nn.linear(512, 256),       #512 u 256
            nn.relu(),
            nn.dropout(0.4),
            nn.linear(256, num_classes) #256 u broj emocija
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.2),
])

val_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

In [ ]:
import zipfile, os
zip_path = 'archive.zip'
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('./')

In [ ]:
#ucitavanje podataka
train_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)
val_dataset   = datasets.ImageFolder(DATA_DIR, transform=val_transform)

#velicina splitova
train_count = int(len(train_dataset) * (1 - VAL_SPLIT))
val_count   = len(train_dataset) - train_count

#split za train
generator = torch.Generator().manual_seed(42)
train_data, _ = random_split(train_dataset, [train_count, val_count], generator=generator)

#split za val
generator = torch.Generator().manual_seed(42)
_, val_data   = random_split(val_dataset,   [train_count, val_count], generator=generator)

class_names = train_dataset.classes
num_class   = len(class_names)
print(f"Classes : {class_names}")
print(f"Train   : {train_count}  |  Val : {val_count}")

In [ ]:
train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True,  num_workers=12, pin_memory=True)
val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False, num_workers=12, pin_memory=True)

In [ ]:
#model, loss i optimizator
model     = CNN(num_classes=num_class).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE)

#scheduler za lr
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=5, factor=0.5, verbose=True)

#scaler za fp16
scaler = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

In [ ]:
def count_parameters(model):
    total     = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Total parameters    : {total:,}")
    print(f"Trainable parameters: {trainable:,}")

count_parameters(model)

In [ ]:
def train():
    #inicijalizacija
    best_val_acc     = 0.0
    patience_counter = 0
    history          = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}

    for epoch in range(1, EPOCHS + 1):
        #trening faza
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        t0 = time.time()

        for images, labels in train_loader:
            images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
            optimizer.zero_grad()

            #prolaz u fp16 preciznosti
            with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                outputs = model(images)
                loss    = criterion(outputs, labels)

            #backprop sa scalerom
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += loss.item() * images.size(0)
            correct      += (outputs.detach().argmax(dim=1) == labels).sum().item()
            total        += images.size(0)

        avg_loss = running_loss / total
        avg_acc  = correct / total

        #validacijska faza
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device, non_blocking=True), labels.to(device, non_blocking=True)
                with torch.autocast(device_type=device.type, enabled=torch.cuda.is_available()):
                    outputs = model(images)
                    loss    = criterion(outputs, labels)
                val_loss    += loss.item() * images.size(0)
                val_correct += (outputs.argmax(dim=1) == labels).sum().item()
                val_total   += images.size(0)

        val_loss /= val_total
        val_acc   = val_correct / val_total
        elapsed   = time.time() - t0

        #ispis rezultata epohe
        print(f"EPOCH {epoch:3}/{EPOCHS}  "
              f"train_loss={avg_loss:.3f} train_acc={avg_acc:.3f}  "
              f"val_loss={val_loss:.3f} val_acc={val_acc:.3f}  "
              f"lr={optimizer.param_groups[0]['lr']:.2e}  {elapsed:.1f}s")

        history["train_loss"].append(avg_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(avg_acc)
        history["val_acc"].append(val_acc)

        #azuriranje lr-a
        scheduler.step(val_loss)

        #provjera i spremanje najboljeg modela
        if val_acc > best_val_acc:
            best_val_acc     = val_acc
            patience_counter = 0
            torch.save({
                "model_state": model.state_dict(),
                "class_names": class_names,
                "img_size"   : IMG_SIZE,
            }, MODEL_PATH)
            print(f"  ✓ Saved best model  val_acc={best_val_acc:.3f}")
        else:
            #early stopping provjera
            patience_counter += 1
            if patience_counter >= PATIENCE:
                print(f"Early stopping triggered at epoch {epoch} "
                      f"(no improvement for {PATIENCE} epochs)")
                break

    print("Training complete.")
    return history

In [ ]:
def plot_history(history):
    #inicijalizacija grafova
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    #graf za loss
    ax1.plot(history["train_loss"], label="Train Loss")
    ax1.plot(history["val_loss"],   label="Val Loss")
    ax1.set_title("Loss"); ax1.set_xlabel("Epoch"); ax1.legend()

    #graf za tocnost
    ax2.plot(history["train_acc"], label="Train Acc")
    ax2.plot(history["val_acc"],   label="Val Acc")
    ax2.set_title("Accuracy"); ax2.set_xlabel("Epoch"); ax2.legend()

    #prikaz i spremanje
    plt.tight_layout()
    plt.savefig("training_curves.png", dpi=150)
    plt.show()

In [ ]:
def evaluate(model, loader, class_names):
    """Print classification report + confusion matrix heatmap."""
    model.eval()
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images  = images.to(device)
            outputs = model(images)
            preds   = outputs.argmax(dim=1).cpu()
            all_preds.extend(preds.numpy())
            all_labels.extend(labels.numpy())

    print(classification_report(all_labels, all_preds, target_names=class_names))

    cm = confusion_matrix(all_labels, all_preds)
    plt.figure(figsize=(8, 6))
    sns.heatmap(cm, annot=True, fmt="d",
                xticklabels=class_names, yticklabels=class_names,
                cmap="Blues")
    plt.xlabel("Predicted"); plt.ylabel("Actual")
    plt.title("Confusion Matrix")
    plt.tight_layout()
    plt.savefig("confusion_matrix.png", dpi=150)
    plt.show()

In [ ]:
history = train()
plot_history(history)

checkpoint = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(checkpoint["model_state"])
evaluate(model, val_loader, class_names)

In [ ]:
#klasifikator za lica
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + "haarcascade_frontalface_default.xml"
)

def inference():
    #ucitavanje modela
    checkpoint = torch.load(MODEL_PATH, map_location=device)
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    model.eval()
    names = checkpoint["class_names"]

    #pokretanje kamere
    print("Webcam starting; press q to quit")
    webcam = cv2.VideoCapture(0)
    if not webcam.isOpened():
        print("Webcam error"); return

    while True:
        ret, frame = webcam.read()
        if not ret:
            break

        #detekcija lica
        gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5)

        if len(faces) == 0:
            #klasifikacija cijelog okvira ako nema lica
            rgb    = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            tensor = val_transform(Image.fromarray(rgb)).unsqueeze(0).to(device)
            with torch.no_grad():
                probs      = torch.softmax(model(tensor), dim=1)
                conf, idx  = torch.max(probs, dim=1)
            cv2.putText(frame, f"{names[idx.item()]} {conf.item():.2f} (no face)",
                        (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
        else:
            #klasifikacija svakog detektiranog lica
            for (x, y, w, h) in faces:
                face_crop = frame[y:y+h, x:x+w]
                rgb       = cv2.cvtColor(face_crop, cv2.COLOR_BGR2RGB)
                tensor    = val_transform(Image.fromarray(rgb)).unsqueeze(0).to(device)

                with torch.no_grad():
                    probs      = torch.softmax(model(tensor), dim=1)
                    conf, idx  = torch.max(probs, dim=1)
                    label      = names[idx.item()]

                #crtanje okvira i teksta
                cv2.rectangle(frame, (x, y), (x+w, y+h), (255, 0, 0), 2)
                cv2.putText(frame, f"{label} {conf.item():.2f}",
                            (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 0, 0), 2)

        #prikaz i izlaz
        cv2.imshow("Emotion Detector (q to quit)", frame)
        if cv2.waitKey(1) & 0xFF == ord("q"):
            break

    #gasenje kamere
    webcam.release()
    cv2.destroyAllWindows()

In [ ]:
inference()